<a href="https://colab.research.google.com/github/vedikaa26/OOPS/blob/main/Music_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/openai/whisper.git
!pip install yt-dlp


  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-0c8dbf__
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-0c8dbf__
  Resolved https://github.com/openai/whisper.git to commit 04f449b8a437f1bbd3dba5c9f826aca972e7709a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=3b3ab24e94002873a6c1407dd62c4785018dfa828aaa675ddfa70f230cdc0fa4
  Stored in directory: /tmp/pip-ephem-wheel-cache-u1_spv6h/wheels/c3/03/25/5e0ba78bc27a3a089f137c9f1d92fdfce16d06996c071a016c
Successfully built openai-whisper
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.6 MB/s eta 0:00:00


In [ ]:
import yt_dlp
import whisper

# 1. The Target Video
# Using a classic folk tune for our first test run!
youtube_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ" # Changed to a different, widely available video
audio_file = "audio.m4a"

# 2. Download the audio track using yt-dlp
print("Downloading audio track...")
ydl_opts = {
    'format': 'm4a/bestaudio/best',
    'outtmpl': audio_file,
    'quiet': True # Keeps the output looking clean
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([youtube_url])

# 3. Load the AI and Transcribe
print("Loading Whisper model... (Using the 'base' model for speed)")
model = whisper.load_model("base")

print("Listening and transcribing... this takes a few seconds.")
result = model.transcribe(audio_file)

# 4. Show the results!
print("\n--- FULL TRANSCRIPT ---")
print(result["text"])

Loading Whisper model... (Using the 'base' model for speed)


100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 61.4MiB/s]


Listening and transcribing... this takes a few seconds.

--- FULL TRANSCRIPT ---
 Music There are no strangers to love You know the rules and something hard The commitments fall out, keep getting up You won't get this wrong any other guy I just wanna tell you how I feel I gotta make you understand Never gonna give you up Never gonna let you down Never gonna run around it He's ready to Never gonna make you cry Never gonna say goodbye Never gonna tell the lie and hurt you We've known each other for so long Your heart's been aching but You're too shy to say it It's as if we've all know what's been going on We know the game I'm with I'm gonna play it And if you ask me how I feel I gotta make you understand Never gonna give you up Never gonna let you down Never gonna run around it He's ready to Never gonna make you cry Never gonna say goodbye Never gonna tell the lie and hurt you Never gonna give you up Never gonna let you down Never gonna run around it He's ready to Never gonna make you cr

In [ ]:
import yt_dlp
import whisper
import datetime

# 1. The Target Video (Testing with a Hindi video clip)
youtube_url = "https://www.youtube.com/watch?v=dQw4w9WgXcQ" # Changed to a widely available, known working video
audio_file = "multilingual_audio.m4a"

# 2. Clean Download
print("Downloading audio track...")
ydl_opts = {
    'format': 'm4a/bestaudio/best',
    'outtmpl': audio_file,
    'overwrites': True, # Overwrite previous file if it exists
    'quiet': True
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([youtube_url])

# 3. Load the Model
print("Loading Whisper model...")
model = whisper.load_model("base")

# 4. Transcribe WITH Translation Task
print("Processing audio (Translating to English and calculating timestamps)...")
# Setting task='translate' forces Whisper to output English text, regardless of the audio language
result = model.transcribe(audio_file, task="translate")

# 5. Helper function to format seconds into clean HH:MM:SS timestamps
def format_time(seconds):
    td = datetime.timedelta(seconds=seconds)
    # Formats the timedelta string to look like HH:MM:SS
    return str(td).split('.')[0].zfill(8)

# 6. Show the Structured Output
print("\n" + "="*50)
print("       TRANSLATED TRANSCRIPT WITH TIMESTAMPS")
print("="*50 + "\n")

for segment in result["segments"]:
    start = format_time(segment["start"])
    end = format_time(segment["end"])
    text = segment["text"].strip()

    print(f"[{start} --> {end}] {text}")

Loading Whisper model...
Processing audio (Translating to English and calculating timestamps)...

       TRANSLATED TRANSCRIPT WITH TIMESTAMPS

[00:00:00 --> 00:00:16] Somebody tässä!
[00:00:30 --> 00:00:39] You won't get this wrong any other guy I just want to tell you how I feel
[00:00:39 --> 00:00:47] I got to make you understand Never gonna give you up, never gonna let you down
[00:00:47 --> 00:00:53] Never gonna run around it, he's ready to Never gonna make you cry
[00:00:53 --> 00:00:59] Never gonna say goodbye Never gonna tell the lie and hurt you
[00:00:59 --> 00:01:07] We've known each other for so long Your heart's been aching back
[00:01:07 --> 00:01:13] You're too shy to say it Is how we've all known what's been gone on
[00:01:13 --> 00:01:22] We know the game I'm with Gonna play it And if you ask me how I feel
[00:01:22 --> 00:01:29] Don't tell me you're too bad to see Never gonna give you up, never gonna let you down
[00:01:29 --> 00:01:35] Never gonna run around it, he's

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import yt_dlp
import whisper
import datetime
import os

# 1. Load the model once globally (so it doesn't reload every time someone clicks submit)
print("Loading Whisper model into memory...")
model = whisper.load_model("base")

# Helper function for timestamps
def format_time(seconds):
    td = datetime.timedelta(seconds=seconds)
    return str(td).split('.')[0].zfill(8)

# 2. The Core Pipeline Function
def process_youtube_video(youtube_url):
    audio_file = "webapp_audio.m4a"

    # Download
    ydl_opts = {
        'format': 'm4a/bestaudio/best',
        'outtmpl': audio_file,
        'overwrites': True,
        'quiet': True
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([youtube_url])
    except Exception as e:
        return f"Error downloading video. Make sure the URL is correct! Details: {e}"

    # Transcribe & Translate
    result = model.transcribe(audio_file, task="translate")

    # Format the output script
    formatted_script = ""
    for segment in result["segments"]:
        start = format_time(segment["start"])
        end = format_time(segment["end"])
        text = segment["text"].strip()
        formatted_script += f"[{start} --> {end}] {text}\n"

    return formatted_script

# 3. Build the Gradio UI
demo = gr.Interface(
    fn=process_youtube_video,
    inputs=gr.Textbox(
        label="YouTube URL",
        placeholder="Drop a link here..."
    ),
    outputs=gr.Textbox(
        label="Translated & Timestamped Transcript",
        lines=15,
        show_copy_button=True
    ),
    title="🎙️ Multilingual Video Transcriber",
    description="Powered by OpenAI Whisper. Drop a video link below to extract the audio, translate it to English, and generate a timestamped script.",
    theme=gr.themes.Soft(),
    allow_flagging="never"
)

# 4. Launch the public app!
demo.launch(share=True)

Loading Whisper model into memory...


/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://736284460431695856.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import yt_dlp
import whisper
import datetime
import os
from collections import Counter
import re

print("Loading Whisper model into memory...")
model = whisper.load_model("base")

# Helper function to format time for SRT files (SRT uses HH:MM:SS,mmm format)
def format_srt_time(seconds):
    td = datetime.timedelta(seconds=seconds)
    time_str = str(td).zfill(8)
    if '.' in time_str:
        base, ms = time_str.split('.')
        ms = ms[:3].ljust(3, '0')
    else:
        base, ms = time_str, '000'
    return f"{base},{ms}"

def format_clean_time(seconds):
    td = datetime.timedelta(seconds=seconds)
    return str(td).split('.')[0].zfill(8)

# Simple NLP helper to extract key terms (ignoring common filler words)
def extract_keywords(text):
    stop_words = set(['the', 'a', 'and', 'to', 'in', 'is', 'you', 'that', 'it', 'he', 'was', 'for', 'on', 'are', 'as', 'with', 'his', 'they', 'i', 'this', 'of', 'at', 'by', 'not', 'but', 'from'])
    words = re.findall(r'\b\w+\b', text.lower())
    important_words = [w for w in words if w not in stop_words and len(w) > 3]
    counts = Counter(important_words).most_common(7)

    if not counts:
        return "No significant keywords found."

    return "\n".join([f"🔑 {word.title()}: Mentioned {count} times" for word, count in counts])

# The Ultimate Pipeline Function
def process_ultimate_transcriber(youtube_url):
    audio_file = "final_app_audio.m4a"
    srt_file_path = "subtitles.srt"

    # 1. Download Audio
    ydl_opts = {
        'format': 'm4a/bestaudio/best',
        'outtmpl': audio_file,
        'overwrites': True,
        'quiet': True,
        'http_headers': {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        },
        'cookiefile': 'cookies.txt' # Add this line, make sure you upload a cookies.txt file to Colab
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([youtube_url])
    except Exception as e:
        return f"Error downloading video: {e}", None, "Error processing analytics."

    # 2. AI Inference
    result = model.transcribe(audio_file, task="translate")

    # 3. Generate both a readable script AND a formal SRT file structure
    readable_script = ""
    srt_content = ""

    for idx, segment in enumerate(result["segments"], start=1):
        start_secs = segment["start"]
        end_secs = segment["end"]
        text = segment["text"].strip()

        # Readable layout
        readable_script += f"[{format_clean_time(start_secs)} --> {format_clean_time(end_secs)}] {text}\n"

        # Valid SRT layout
        srt_content += f"{idx}\n"
        srt_content += f"{format_srt_time(start_secs)} --> {format_srt_time(end_secs)}\n"
        srt_content += f"{text}\n\n"

    # Write the SRT file to disk so user can download it
    with open(srt_file_path, "w", encoding="utf-8") as f:
        f.write(srt_content)

    # 4. Extract Analytics
    analytics = extract_keywords(result["text"])

    return readable_script, srt_file_path, analytics

# Building a multi-tab interface using Gradio Blocks
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ Production-Grade Video Intelligence & Subtitle Suite")
    gr.Markdown("Drop any YouTube link to instantly generate English transcripts, export industry-standard subtitle files, and pull content analytics.")

    with gr.Row():
        url_input = gr.Textbox(label="YouTube Video Link", placeholder="Paste URL here (Hindi, French, English, etc.)...")

    submit_btn = gr.Button("Analyze & Transcribe", variant="primary")

    with gr.Tabs():
        with gr.TabItem("📄 Interactive Transcript"):
            script_output = gr.Textbox(label="Timestamped Script", lines=12, show_copy_button=True)

        with gr.TabItem("💾 Export Subtitles"):
            gr.Markdown("### Download `.srt` file for video editors (Premiere Pro, YouTube Studio, VLC)")
            file_output = gr.File(label="Download Subtitle File")

        with gr.TabItem("📊 Content Analytics"):
            gr.Markdown("### Key Topics & Term Frequency")
            analytics_output = gr.Textbox(label="Top Extracted Keywords", lines=6)

    # Tie components together
    submit_btn.click(
        fn=process_ultimate_transcriber,
        inputs=[url_input],
        outputs=[script_output, file_output, analytics_output]
    )

demo.launch(share=True)

Loading Whisper model into memory...


/tmp/ipykernel_2787/1781629670.py:91: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ad34121da5bc05c229.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import yt_dlp
import whisper
import datetime
import os
import uuid
from collections import Counter
import re

print("Loading Whisper model into memory...")
model = whisper.load_model("base")

# Helper function to format time for SRT files
def format_srt_time(seconds):
    td = datetime.timedelta(seconds=seconds)
    time_str = str(td).zfill(8)
    if '.' in time_str:
        base, ms = time_str.split('.')
        ms = ms[:3].ljust(3, '0')
    else:
        base, ms = time_str, '000'
    return f"{base},{ms}"

def format_clean_time(seconds):
    td = datetime.timedelta(seconds=seconds)
    return str(td).split('.')[0].zfill(8)

# Simple NLP helper
def extract_keywords(text):
    stop_words = set(['the', 'a', 'and', 'to', 'in', 'is', 'you', 'that', 'it', 'he', 'was', 'for', 'on', 'are', 'as', 'with', 'his', 'they', 'i', 'this', 'of', 'at', 'by', 'not', 'but', 'from'])
    words = re.findall(r'\b\w+\b', text.lower())
    important_words = [w for w in words if w not in stop_words and len(w) > 3]
    counts = Counter(important_words).most_common(7)

    if not counts:
        return "No significant keywords found."
    return "\n".join([f"🔑 {word.title()}: Mentioned {count} times" for word, count in counts])

# The Bulletproof Pipeline Function
def process_ultimate_transcriber(youtube_url, uploaded_file):
    # 1. Generate unique IDs to prevent file overwrite bugs (The Concurrency Fix)
    unique_id = str(uuid.uuid4())
    srt_file_path = f"subtitles_{unique_id}.srt"
    audio_path = None

    # 2. Routing Logic: Did they upload a file, or paste a link?
    if uploaded_file is not None:
        # User uploaded a file directly! We bypass YouTube completely.
        audio_path = uploaded_file

    elif youtube_url and youtube_url.strip() != "":
        # User provided a link, let's try to download it
        audio_path = f"audio_{unique_id}.m4a"
        ydl_opts = {
            'format': 'm4a/bestaudio/best',
            'outtmpl': audio_path,
            'overwrites': True,
            'quiet': True,
            # 'cookiefile': 'cookies.txt' # Uncomment this line once you upload a cookies.txt file to Colab!
        }
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([youtube_url])
        except Exception as e:
            return f"YouTube Blocked the Request (Error 429). \n\nDetails: {e}\n\nPlease use the manual Audio Upload tool instead!", None, "Error processing analytics."
    else:
        return "Please provide a YouTube link or upload an audio file.", None, "No input."

    # 3. AI Inference (Works the same whether it came from YouTube or an upload)
    result = model.transcribe(audio_path, task="translate")

    # 4. Generate outputs
    readable_script = ""
    srt_content = ""

    for idx, segment in enumerate(result["segments"], start=1):
        start_secs = segment["start"]
        end_secs = segment["end"]
        text = segment["text"].strip()

        readable_script += f"[{format_clean_time(start_secs)} --> {format_clean_time(end_secs)}] {text}\n"

        srt_content += f"{idx}\n"
        srt_content += f"{format_srt_time(start_secs)} --> {format_srt_time(end_secs)}\n"
        srt_content += f"{text}\n\n"

    with open(srt_file_path, "w", encoding="utf-8") as f:
        f.write(srt_content)

    analytics = extract_keywords(result["text"])

    return readable_script, srt_file_path, analytics

# The Updated UI with Side-by-Side Inputs
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ Resilient Video & Audio Transcriber")
    gr.Markdown("Extract subtitles and analytics from a YouTube link, or upload an audio file directly if YouTube is rate-limiting the server.")

    with gr.Row():
        with gr.Column(scale=2):
            url_input = gr.Textbox(label="Option 1: YouTube Video Link", placeholder="Paste URL here...")
        with gr.Column(scale=1):
            file_input = gr.Audio(type="filepath", label="Option 2: Upload Audio File directly")

    submit_btn = gr.Button("Analyze & Transcribe", variant="primary")

    with gr.Tabs():
        with gr.TabItem("📄 Interactive Transcript"):
            script_output = gr.Textbox(label="Timestamped Script", lines=12, show_copy_button=True)

        with gr.TabItem("💾 Export Subtitles"):
            gr.Markdown("### Download `.srt` file")
            file_output = gr.File(label="Download Subtitle File")

        with gr.TabItem("📊 Content Analytics"):
            analytics_output = gr.Textbox(label="Top Extracted Keywords", lines=6)

    # Tie components together (Notice we pass both inputs now!)
    submit_btn.click(
        fn=process_ultimate_transcriber,
        inputs=[url_input, file_input],
        outputs=[script_output, file_output, analytics_output]
    )

demo.launch(share=True)

Loading Whisper model into memory...


/tmp/ipykernel_2787/4128464463.py:95: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://57321b41e74f83c714.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
